# Learning 3: Simple Chains

**Goal**: Connect components together using the LCEL pipe operator

## What You'll Learn
- What chains are and why they're useful
- Using the `|` pipe operator
- Chaining prompts, LLMs, and output parsers
- Creating multi-step chains

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")
print("Setup complete!")

## What is a Chain?

A chain connects multiple components:

```
[Input] → [Prompt] → [LLM] → [Output Parser] → [Result]
```

Instead of calling each component separately, chains let you combine them into a single callable.

## The Pipe Operator `|`

LangChain uses the pipe operator `|` to chain components. It's called LCEL (LangChain Expression Language).

In [ ]:
# Without chaining (verbose)
prompt = ChatPromptTemplate.from_template("Tell me a fact about {topic}")
formatted = prompt.format_messages(topic="dogs")
response = llm.invoke(formatted)
text = response.content

print("Without chain:", text)

In [ ]:
# With chaining (clean!)
prompt = ChatPromptTemplate.from_template("Tell me a fact about {topic}")
output_parser = StrOutputParser()

chain = prompt | llm | output_parser

result = chain.invoke({"topic": "cats"})
print("With chain:", result)

## Understanding StrOutputParser

The `StrOutputParser` extracts just the text content from the LLM response.

In [ ]:
# Without parser - returns AIMessage object
chain_no_parser = prompt | llm
result = chain_no_parser.invoke({"topic": "birds"})
print("Type:", type(result))
print("Result:", result)

In [ ]:
# With parser - returns just the string
chain_with_parser = prompt | llm | StrOutputParser()
result = chain_with_parser.invoke({"topic": "birds"})
print("Type:", type(result))
print("Result:", result)

## Multi-Step Chain

Chain the output of one LLM call as input to another.

In [ ]:
# Step 1: Generate a topic
topic_prompt = ChatPromptTemplate.from_template(
    "Give me a random {category} topic in 3 words or less. Just the topic, nothing else."
)

# Step 2: Write about that topic
story_prompt = ChatPromptTemplate.from_template(
    "Write a 2-sentence story about: {topic}"
)

# Chain step 1
topic_chain = topic_prompt | llm | StrOutputParser()

# Test step 1
topic = topic_chain.invoke({"category": "science"})
print("Generated topic:", topic)

In [ ]:
# Combine both chains using RunnablePassthrough
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Full pipeline
full_chain = (
    {"topic": topic_prompt | llm | StrOutputParser()}
    | story_prompt
    | llm
    | StrOutputParser()
)

result = full_chain.invoke({"category": "history"})
print("Story:", result)

## Parallel Chains with RunnableParallel

Run multiple chains at once and combine their outputs.

In [ ]:
from langchain_core.runnables import RunnableParallel

# Create parallel chains
joke_prompt = ChatPromptTemplate.from_template("Tell a short joke about {topic}")
fact_prompt = ChatPromptTemplate.from_template("Tell a fact about {topic}")

parallel_chain = RunnableParallel(
    joke=joke_prompt | llm | StrOutputParser(),
    fact=fact_prompt | llm | StrOutputParser()
)

result = parallel_chain.invoke({"topic": "coffee"})
print("Joke:", result["joke"])
print("\nFact:", result["fact"])

## Adding Custom Functions with RunnableLambda

In [ ]:
from langchain_core.runnables import RunnableLambda

# Custom function to process output
def make_uppercase(text: str) -> str:
    return text.upper()

def add_emoji(text: str) -> str:
    return f"✨ {text} ✨"

# Chain with custom functions
chain = (
    ChatPromptTemplate.from_template("Say hello to {name}")
    | llm
    | StrOutputParser()
    | RunnableLambda(make_uppercase)
    | RunnableLambda(add_emoji)
)

result = chain.invoke({"name": "Alice"})
print(result)

## Streaming with Chains

Chains support streaming too!

In [ ]:
chain = (
    ChatPromptTemplate.from_template("Write a haiku about {topic}")
    | llm
    | StrOutputParser()
)

print("Streaming:")
for chunk in chain.stream({"topic": "programming"}):
    print(chunk, end="", flush=True)

## Exercise: Build Your Own Chain

1. Create a chain that translates text and then summarizes it
2. Create parallel chains for different types of analysis
3. Add a custom function to your chain

In [ ]:
# Your code here!



## Key Takeaways

1. Use `|` to chain components together
2. `StrOutputParser()` extracts text from LLM responses
3. `RunnableParallel` runs multiple chains simultaneously
4. `RunnableLambda` wraps custom functions for use in chains
5. Chains support `.invoke()`, `.stream()`, and `.batch()`

**Next**: Learning 4 - Tools Basics